# 04 — Sorting Algorithms

## Why This Matters for DSA
Sorting closes out Phase 02 because it's the technique that CREATES the structure earlier notebooks assumed was already there — two pointers and binary search both required sortedness, and now you'll see exactly how that sortedness gets produced, at what cost, and with what guarantees. Beyond that, sorting algorithms are the cleanest possible teaching example of real complexity trade-offs: same problem, half a dozen genuinely different solutions, each with a different time/space/stability profile. Understanding WHY `std::sort` picks the strategy it does is worth more than memorizing any single algorithm's code.

## Prerequisites
- `01_Complexity_Analysis` (Phase 01) — recurrence relations, best/average/worst case
- `02_Recursion` (Phase 01) — divide and conquer is a direct application of the recursion patterns from that notebook
- `03_STL_Deep_Dive` (Phase 00) — heap mechanics (Section 5.6), referenced directly in heap sort

## Learning Objectives
By the end of this notebook, you will be able to:
- Implement bubble, selection, and insertion sort, and explain why their best-case behaviors differ despite identical worst-case complexity
- Implement merge sort and explain why it guarantees O(n log n) in every case, at the cost of O(n) extra space
- Implement quicksort, demonstrate its O(n²) worst case empirically, and explain the pivot-choice fix
- Implement heap sort and explain its "best of both worlds" complexity profile, and why it's rarely the fastest in practice anyway
- Implement counting sort and radix sort, and explain how they break the O(n log n) comparison-sort lower bound
- Define sorting stability precisely, identify which algorithms provide it, and know when it actually matters
- Choose the right sort for a given situation instead of defaulting to `std::sort` without thinking about it

## Checklist Before Moving On
- [ ] I can explain why insertion sort is the practical choice for small or nearly-sorted arrays despite being O(n²) worst case
- [ ] I can state merge sort's recurrence relation and explain how it resolves to O(n log n)
- [ ] I can demonstrate quicksort's worst case on a concrete adversarial input and explain the randomized-pivot fix
- [ ] I understand why heap sort's strong worst-case guarantee doesn't automatically make it the fastest choice
- [ ] I can explain the structural requirement that lets counting/radix sort beat the comparison-sort lower bound
- [ ] I can define stability precisely and know which sorts provide it without checking

## Table of Contents
1. The O(n²) Family — Bubble, Selection, and Insertion Sort
2. Merge Sort — Divide and Conquer With a Guarantee
3. Quick Sort — Partitioning, and Why the Pivot Choice Matters
4. Heap Sort — O(n log n) Guaranteed, In Place
5. Counting Sort — Breaking the Comparison-Sort Lower Bound
6. Radix Sort — Extending Counting Sort to Larger Ranges
7. Sorting Stability — What It Means and When It Matters
8. Choosing the Right Sort
9. Phase Checkpoint, Cheat Sheet, and Answer Key
10. LeetCode Practice Problems


## 1. The O(n²) Family — Bubble, Selection, and Insertion Sort

### The Why
All three of these are O(n²) worst case, and none of them is what you'd reach for on a large dataset — but they're not purely academic. Insertion sort specifically is the real-world choice for small or nearly-sorted data, and understanding WHY exposes the difference between "worst-case complexity" and "practical performance" more clearly than almost any other comparison in this course.

### Core Mechanism
- **Bubble sort:** repeatedly swap adjacent out-of-order elements; each full pass "bubbles" the largest unsorted element to its correct position at the end. The early-exit optimization (stop if a pass makes zero swaps) gives it an O(n) BEST case on already-sorted input.
- **Selection sort:** repeatedly scan the unsorted remainder for its minimum, swap it into place. This ALWAYS scans the full remainder, even on already-sorted input — no early exit is possible, so it's O(n²) in every case, best included. Its one advantage: exactly one swap per outer iteration, useful when swaps are expensive relative to comparisons.
- **Insertion sort:** build a sorted prefix one element at a time, shifting larger elements right to make room for each new insertion. Best case (already sorted) is O(n) — the inner while loop never executes. Critically, it stays FAST on NEARLY-sorted data too, since each element only needs to shift a short distance — this is why insertion sort is the practical choice for small arrays or data that's mostly-already-sorted, and why many real `std::sort` implementations (including introsort, Section 8) switch to insertion sort for small sub-arrays internally.

### Common Pitfalls
- Assuming "O(n²) worst case" means "always slow" — insertion sort's near-sorted performance is a genuine, frequently-exploited practical advantage, not a technicality.
- Forgetting selection sort's lack of an early exit — unlike bubble and insertion sort, no amount of pre-sortedness in the input speeds it up.


In [ ]:
%%writefile temp.cpp
#include <iostream>
#include <vector>
using namespace std;

// BUBBLE SORT: repeatedly swap adjacent out-of-order elements. After each full pass,
// the largest unsorted element "bubbles up" to its correct final position at the end.
void bubbleSort(vector<int> v) {
    int n = (int)v.size();
    for (int i = 0; i < n - 1; i++) {
        bool swapped = false;
        for (int j = 0; j < n - 1 - i; j++) {   // shrinking range: the last i elements are
                                                   // already confirmed sorted, skip re-checking them
            if (v[j] > v[j + 1]) {
                swap(v[j], v[j + 1]);
                swapped = true;
            }
        }
        if (!swapped) break;   // OPTIMIZATION: if a full pass made no swaps, the array is
                                 // already sorted -- stop early instead of doing n more useless passes
    }
    cout << "bubble: ";
    for (int x : v) cout << x << " ";
    cout << "\n";
}

// SELECTION SORT: repeatedly find the MINIMUM of the unsorted remainder, swap it into place.
// Unlike bubble sort, this does exactly one swap per outer iteration -- useful when swaps
// are expensive (e.g. large objects) even though comparisons are the same O(n^2).
void selectionSort(vector<int> v) {
    int n = (int)v.size();
    for (int i = 0; i < n - 1; i++) {
        int minIdx = i;
        for (int j = i + 1; j < n; j++) {
            if (v[j] < v[minIdx]) minIdx = j;
        }
        swap(v[i], v[minIdx]);   // exactly ONE swap per outer iteration, regardless of n
    }
    cout << "selection: ";
    for (int x : v) cout << x << " ";
    cout << "\n";
}

// INSERTION SORT: build up a sorted prefix one element at a time, inserting each new
// element into its correct position within that sorted prefix by shifting larger elements right.
void insertionSort(vector<int> v) {
    int n = (int)v.size();
    for (int i = 1; i < n; i++) {
        int key = v[i];
        int j = i - 1;
        while (j >= 0 && v[j] > key) {   // shift elements right to make room for `key`
            v[j + 1] = v[j];
            j--;
        }
        v[j + 1] = key;
    }
    cout << "insertion: ";
    for (int x : v) cout << x << " ";
    cout << "\n";
}

int main() {
    vector<int> nums{5, 2, 9, 1, 5, 6};
    bubbleSort(nums);
    selectionSort(nums);
    insertionSort(nums);

    // ALL THREE are O(n^2) worst case -- but their BEST cases differ meaningfully:
    // bubble sort: O(n) best case (already sorted, one no-swap pass, early exit)
    // selection sort: ALWAYS O(n^2), even on already-sorted input -- it always scans the
    //                 full remainder to confirm the minimum, no early exit is possible
    // insertion sort: O(n) best case (already sorted, inner while loop never executes) --
    //                 and stays FAST on NEARLY-sorted data too, since each element only
    //                 needs to shift a short distance -- this is why insertion sort is the
    //                 real-world choice for small or nearly-sorted arrays, and why
    //                 std::sort switches to it for small sub-arrays internally

    return 0;
}


In [ ]:
!g++ -std=c++17 -O2 -Wall -o temp temp.cpp && ./temp

## 2. Merge Sort — Divide and Conquer With a Guarantee

### The Why
Merge sort is the algorithm that GUARANTEES O(n log n) no matter what the input looks like — no adversarial input can degrade it, unlike quicksort. That guarantee has a real cost (extra memory), which is exactly the kind of trade-off worth understanding rather than memorizing.

### Core Mechanism
- Recursively divide the array in half until pieces are trivially sorted (size 0 or 1), then MERGE sorted halves back together — the merge step is exactly the "merge two sorted arrays" pattern from `02_Sliding_Window_and_Prefix_Sum`.
- **The recurrence:** `T(n) = 2·T(n/2) + O(n)` — two half-size recursive calls, plus O(n) merge work at each level. This resolves to O(n log n): there are `log n` levels of recursion (each halving until reaching size 1), and the total merge work across any ONE level is O(n) (every element gets touched exactly once per level, combined across all the merges happening at that level).
- **The guarantee holds in ALL cases** — best, average, and worst — because the divide step always splits evenly regardless of the data's actual values; there's no data-dependent behavior for an adversary to exploit.
- **The cost:** O(n) extra space for the temporary arrays used during merging — this is NOT an in-place sort, unlike quicksort or heap sort.
- **Naturally stable:** using `<=` (not `<`) in the merge comparison means that when the left and right elements are equal, the LEFT one is placed first — preserving original relative order for equal elements.

### Common Pitfalls
- Using `<` instead of `<=` in the merge comparison "because it seems equivalent" — this flips stability; with strict `<`, the right element gets placed first on ties, silently breaking the stability guarantee.
- Underestimating the O(n) extra-space cost for very large arrays, where memory (not just time) can be the actual constraint.


In [ ]:
%%writefile temp.cpp
#include <iostream>
#include <vector>
using namespace std;

// MERGE SORT: divide the array in half recursively until pieces are trivially sorted
// (size 0 or 1), then MERGE sorted halves back together -- the merge step is exactly
// the "merge two sorted arrays" pattern from 02_Sliding_Window_and_Prefix_Sum.

void merge(vector<int>& v, int left, int mid, int right) {
    vector<int> leftPart(v.begin() + left, v.begin() + mid + 1);
    vector<int> rightPart(v.begin() + mid + 1, v.begin() + right + 1);

    int i = 0, j = 0, k = left;
    while (i < (int)leftPart.size() && j < (int)rightPart.size()) {
        if (leftPart[i] <= rightPart[j]) v[k++] = leftPart[i++];
        else v[k++] = rightPart[j++];
    }
    while (i < (int)leftPart.size()) v[k++] = leftPart[i++];
    while (j < (int)rightPart.size()) v[k++] = rightPart[j++];
}

void mergeSort(vector<int>& v, int left, int right) {
    if (left >= right) return;   // base case: 0 or 1 elements, already "sorted"

    int mid = left + (right - left) / 2;
    mergeSort(v, left, mid);        // sort left half
    mergeSort(v, mid + 1, right);   // sort right half
    merge(v, left, mid, right);     // merge the two sorted halves

    // recurrence: T(n) = 2*T(n/2) + O(n) -- two half-size recursive calls, plus O(n)
    // merge work at each level. This solves to O(n log n): log n levels of recursion,
    // O(n) total work at each level (the merges at any one level touch every element
    // exactly once combined)
}

int main() {
    vector<int> nums{5, 2, 9, 1, 5, 6, 3, 8};
    mergeSort(nums, 0, (int)nums.size() - 1);

    cout << "merge sorted: ";
    for (int x : nums) cout << x << " ";
    cout << "\n";

    // GUARANTEES O(n log n) in ALL cases -- best, average, AND worst. No input can make
    // merge sort degrade, unlike quicksort (next section). The cost: O(n) EXTRA space
    // for the temporary left/right arrays during merging -- not in-place, unlike quicksort
    // or heap sort. Also naturally STABLE: leftPart[i] <= rightPart[j] (not <) means equal
    // elements from the left half are placed first, preserving their original relative order

    return 0;
}


In [ ]:
!g++ -std=c++17 -O2 -Wall -o temp temp.cpp && ./temp

## 3. Quick Sort — Partitioning, and Why the Pivot Choice Matters

### The Why
Quicksort is usually FASTER than merge sort in practice (better cache locality, in-place, no extra allocation) — but "usually" is doing real work in that sentence, and this section proves it with a concrete adversarial input rather than just asserting it.

### Core Mechanism
- Pick a pivot, PARTITION the array so everything smaller ends up before it and everything bigger ends up after it (the pivot lands in its final correct position after partitioning), then recursively sort each side. Unlike merge sort, this happens entirely IN PLACE.
- **Lomuto partition scheme** (shown here): the pivot is always the last element of the current range. A boundary index tracks the end of the "confirmed smaller than pivot" region as the loop scans forward, swapping qualifying elements into that region.
- **The empirically demonstrated worst case:** with "always pick the last element" as the pivot strategy, an ALREADY-SORTED input is adversarial — every partition splits into a 0-element side and an (n-1)-element side, the worst possible split. The measured partition-call counts confirm this directly: they scale linearly with n (roughly n-1 calls), not logarithmically — exactly the signature of O(n²) rather than the expected O(n log n).
- **The fix:** choose the pivot RANDOMLY (or via median-of-three: comparing the first, middle, and last elements and picking the median of those three) instead of always the last element. This makes "already sorted" no longer a special adversarial case for the algorithm — the bad split becomes vanishingly unlikely rather than guaranteed, restoring expected O(n log n) behavior in practice.

### Common Pitfalls
- Using a fixed pivot strategy (always first, always last, or anything else predictable) in code that might process sorted or reverse-sorted input — this is precisely the condition that triggers the O(n²) worst case, and it's a realistic input, not a contrived edge case.
- Assuming quicksort is "always O(n log n) in practice" without acknowledging the worst case still genuinely exists and matters for adversarial or already-structured input, hence why merge sort or introsort's fallback (Section 8) exist as guarantees against it.


In [ ]:
%%writefile temp.cpp
#include <iostream>
#include <vector>
#include <cstdlib>
#include <ctime>
using namespace std;

long long partitionCalls = 0;

// QUICK SORT: pick a pivot, PARTITION the array so everything smaller than the pivot
// ends up before it and everything bigger ends up after it, then recursively sort
// each side. Unlike merge sort, partitioning happens IN PLACE -- no extra arrays needed.

// Lomuto partition scheme: pivot is always the LAST element of the current range.
int partition(vector<int>& v, int low, int high) {
    partitionCalls++;
    int pivot = v[high];
    int i = low - 1;   // boundary of "confirmed smaller than pivot" region

    for (int j = low; j < high; j++) {
        if (v[j] < pivot) {
            i++;
            swap(v[i], v[j]);
        }
    }
    swap(v[i + 1], v[high]);   // place the pivot in its final correct position
    return i + 1;               // pivot's final index
}

void quickSort(vector<int>& v, int low, int high) {
    if (low < high) {
        int pivotIdx = partition(v, low, high);
        quickSort(v, low, pivotIdx - 1);
        quickSort(v, pivotIdx + 1, high);
    }
}

int main() {
    vector<int> nums{5, 2, 9, 1, 5, 6, 3, 8};
    quickSort(nums, 0, (int)nums.size() - 1);
    cout << "quicksorted: ";
    for (int x : nums) cout << x << " ";
    cout << "\n";

    // WORST CASE DEMONSTRATION: sorting an ALREADY-SORTED array with this exact pivot
    // choice (always the last element) is quicksort's worst case -- every partition
    // splits into a 0-element side and an (n-1)-element side, giving O(n^2) instead of
    // the O(n log n) average case.
    for (int n : {100, 200, 400}) {
        vector<int> sorted(n);
        for (int i = 0; i < n; i++) sorted[i] = i;   // already sorted -- adversarial input
                                                        // for THIS pivot strategy specifically
        partitionCalls = 0;
        quickSort(sorted, 0, n - 1);
        cout << "n=" << n << " (already sorted) partition calls=" << partitionCalls << "\n";
    }
    // watch partition calls roughly DOUBLE when n doubles -- that's O(n) calls, not
    // O(log n), confirming the O(n^2) worst-case behavior on this adversarial input.
    // THE FIX: choose the pivot RANDOMLY (or via median-of-three) instead of always the
    // last element -- this makes the adversarial "already sorted" input no longer
    // special, restoring expected O(n log n) behavior in practice

    return 0;
}


In [ ]:
!g++ -std=c++17 -O2 -Wall -o temp temp.cpp && ./temp

## 4. Heap Sort — O(n log n) Guaranteed, In Place

### The Why
Heap sort looks like it should be the obvious best choice on paper: O(n log n) guaranteed in every case (like merge sort), AND O(1) extra space (like quicksort). Understanding why it's still not the default choice for general-purpose sorting is a genuinely useful lesson in why Big-O alone doesn't tell the whole performance story.

### Core Mechanism
- Build a max-heap out of the array IN PLACE — same index math as `03_STL_Deep_Dive`'s heap section (parent at `(i-1)/2`, children at `2i+1` and `2i+2`), treating the array itself as the heap's underlying storage.
- **Building the heap:** call `heapify` on every non-leaf node, starting from the LAST non-leaf node and working backward to the root. This bottom-up order guarantees each `heapify` call can assume everything below it is already a valid heap when it runs.
- **Extracting the sorted result:** repeatedly swap the max (always at index 0 in a max-heap) to the end of the still-unsorted region, shrink that region by one, and re-heapify the reduced heap. Doing this n times produces the fully sorted array.
- **The catch:** despite matching or beating merge sort and quicksort on paper (best of both: guaranteed O(n log n) AND O(1) space), heap sort has poor cache locality — its index-math-based access pattern jumps around the array rather than accessing it sequentially, unlike merge sort's linear merges or quicksort's typically-localized partitioning. In practice, this usually makes it slower than a well-implemented quicksort, DESPITE the worse algorithm's better asymptotic worst-case guarantee. This is exactly why `std::sort`'s introsort (Section 8) uses quicksort as its primary strategy, falling back to heap sort only as a safety net when quicksort's recursion depth signals it may be hitting a bad case.

### Common Pitfalls
- Assuming "better worst-case Big-O guarantee" automatically means "faster in practice" — cache locality and constant factors are real, measurable performance effects that Big-O notation deliberately abstracts away, and heap sort is one of the clearest illustrations of that gap.


In [ ]:
%%writefile temp.cpp
#include <iostream>
#include <vector>
using namespace std;

// HEAP SORT: build a max-heap out of the array IN PLACE (treating the array itself as
// the heap's underlying storage, using the same index math as 03_STL_Deep_Dive's heap
// section: parent = (i-1)/2, children = 2i+1, 2i+2), then repeatedly extract the max
// (swap it to the end, shrink the heap, re-heapify) to build the sorted result.

void heapify(vector<int>& v, int n, int i) {
    int largest = i;
    int left = 2 * i + 1, right = 2 * i + 2;

    if (left < n && v[left] > v[largest]) largest = left;
    if (right < n && v[right] > v[largest]) largest = right;

    if (largest != i) {
        swap(v[i], v[largest]);
        heapify(v, n, largest);   // the swap may have violated the heap property further down --
                                    // recurse to fix it
    }
}

void heapSort(vector<int>& v) {
    int n = (int)v.size();

    // build max-heap: heapify every non-leaf node, starting from the LAST non-leaf node
    // working backward to the root -- this bottom-up order guarantees each heapify call
    // can assume everything below it is already a valid heap
    for (int i = n / 2 - 1; i >= 0; i--) {
        heapify(v, n, i);
    }

    // repeatedly extract the max (always at index 0 in a max-heap) by swapping it to
    // the end of the still-unsorted region, shrinking that region, and re-heapifying
    for (int i = n - 1; i > 0; i--) {
        swap(v[0], v[i]);
        heapify(v, i, 0);   // note: n is now `i`, the heap has shrunk by one
    }
}

int main() {
    vector<int> nums{5, 2, 9, 1, 5, 6, 3, 8};
    heapSort(nums);
    cout << "heap sorted: ";
    for (int x : nums) cout << x << " ";
    cout << "\n";

    // O(n log n) GUARANTEED in all cases (like merge sort), but O(1) EXTRA space
    // (like quicksort) -- heap sort is the "best of both worlds" on paper. The catch:
    // it has poor cache locality (heap operations jump around the array via index math,
    // not sequential access), so it's usually SLOWER in practice than a well-implemented
    // quicksort despite the better worst-case guarantee. This is why std::sort's
    // introsort uses quicksort as its primary strategy, falling back to heap sort only
    // when quicksort's recursion depth suggests it's hitting a bad case

    return 0;
}


In [ ]:
!g++ -std=c++17 -O2 -Wall -o temp temp.cpp && ./temp

## 5. Counting Sort — Breaking the Comparison-Sort Lower Bound

### The Why
Every sort covered so far decides ordering by COMPARING elements, and there's a proven mathematical lower bound (Ω(n log n)) on how fast any comparison-based sort can possibly be. Counting sort sidesteps that bound entirely by not comparing elements at all — a genuinely different strategy, not just a faster comparison sort.

### Core Mechanism
- Count how many times each value occurs (a tally array indexed by value), then reconstruct the sorted output directly by emitting each value exactly as many times as it was counted — no comparisons between elements ever happen.
- **Complexity: O(n + k)**, where `k` is the size of the value range — genuinely NOT O(n log n), because there's no comparison-based decision process for the Ω(n log n) lower bound to apply to.
- **The structural requirement that makes this legal:** values must be integers (or map cleanly to small integer keys) within a known, reasonably small range. This ISN'T a loophole in the comparison-sort lower bound — it's a fundamentally different algorithm class that only works under this specific structural assumption.

### Common Pitfalls
- Applying counting sort to a value range that's large relative to `n` (e.g. sorting a small array of values up to 10⁹) — the "O(n+k)" formula looks cheap on paper, but with `k` in the billions, the count array allocation alone is completely impractical, even though the algorithm is technically correct.


In [ ]:
%%writefile temp.cpp
#include <iostream>
#include <vector>
using namespace std;

// COUNTING SORT: breaks the O(n log n) comparison-sort lower bound by NOT comparing
// elements at all -- instead, count how many times each value occurs, then reconstruct
// the sorted output directly from those counts. Only works when values are integers
// in a known, reasonably small range.
vector<int> countingSort(vector<int>& v, int maxVal) {
    vector<int> count(maxVal + 1, 0);
    for (int x : v) count[x]++;               // tally occurrences of each value

    vector<int> result;
    result.reserve(v.size());
    for (int val = 0; val <= maxVal; val++) {
        for (int c = 0; c < count[val]; c++) {  // emit `val`, count[val] times, in order
            result.push_back(val);
        }
    }
    return result;
}

int main() {
    vector<int> nums{4, 2, 2, 8, 3, 3, 1};
    auto sorted = countingSort(nums, 8);
    cout << "counting sorted: ";
    for (int x : sorted) cout << x << " ";
    cout << "\n";

    // COMPLEXITY: O(n + k), where k is the value range (maxVal here). NOT O(n log n) --
    // no comparisons happen at all, which is HOW it breaks the comparison-sort lower
    // bound (that lower bound only applies to sorts that decide order via comparisons).
    // THE CATCH: this is only fast when k is reasonably small relative to n. Sorting
    // values up to 10^9 with counting sort would allocate a count array of a billion
    // entries -- completely impractical despite the "O(n+k)" formula looking cheap on paper

    return 0;
}


In [ ]:
!g++ -std=c++17 -O2 -Wall -o temp temp.cpp && ./temp

## 6. Radix Sort — Extending Counting Sort to Larger Ranges

### The Why
Radix sort answers the practical limitation of counting sort directly: what if the value range is too large for a single counting-sort pass, but the numbers still have bounded WIDTH (a fixed number of digits)? Sort by each digit separately instead of the whole value at once.

### Core Mechanism
- Sort digit by digit, from LEAST significant to MOST significant, using a stable counting sort keyed on just that one digit at each pass.
- **Stability is not optional here — it's what makes the final result correct, not just a nice property.** Consider two numbers sharing the same tens digit but having been placed in a particular relative order by the previous (ones-digit) pass — the tens-digit pass MUST preserve that established relative order for elements that tie on the current digit, or the correctness built up by the earlier pass gets silently destroyed. An unstable per-digit sort produces a genuinely wrong final ordering, not just an unstable-but-still-correct one.
- **Complexity: O(d · (n + k))**, where `d` is the number of digits and `k` is the base (10 for decimal digits). For fixed-width integers, `d` is a constant, making this effectively O(n) — beating the comparison-sort floor the same way counting sort does, just applied digit-by-digit to handle a value range too large for a single counting-sort pass directly.

### Common Pitfalls
- Using an unstable sort for the per-digit passes (implementing it with, say, plain `std::sort` on each digit instead of a stable counting sort) — this silently breaks correctness, not just performance, since later passes depend on earlier passes' relative ordering being preserved.
- Forgetting to determine the correct number of passes needed (based on the maximum value's digit count) — stopping too early leaves higher-order digits unsorted.


In [ ]:
%%writefile temp.cpp
#include <iostream>
#include <vector>
#include <algorithm>
using namespace std;

// RADIX SORT: sorts multi-digit numbers digit by digit, from LEAST significant to MOST
// significant, using a STABLE sort (counting sort by digit) at each pass. Stability is
// NOT optional here -- it's what makes the overall result correct, not just an
// implementation nicety (see the callout below).

int getDigit(int num, int place) {
    return (num / place) % 10;
}

// stable counting sort, keyed on one specific digit (the `place` value: 1, 10, 100, ...)
void countingSortByDigit(vector<int>& v, int place) {
    vector<int> output(v.size());
    vector<int> count(10, 0);

    for (int num : v) count[getDigit(num, place)]++;

    // convert counts to positions: count[d] becomes "how many elements have digit <= d"
    for (int d = 1; d < 10; d++) count[d] += count[d - 1];

    // build output back-to-front -- this specific direction is what keeps the sort STABLE
    // (processing front-to-back here would reverse the relative order of equal-digit elements)
    for (int i = (int)v.size() - 1; i >= 0; i--) {
        int digit = getDigit(v[i], place);
        output[count[digit] - 1] = v[i];
        count[digit]--;
    }

    v = output;
}

void radixSort(vector<int>& v) {
    if (v.empty()) return;
    int maxVal = *max_element(v.begin(), v.end());

    // one pass per digit, starting from the ones place, moving up until `place` exceeds maxVal
    for (long long place = 1; maxVal / place > 0; place *= 10) {
        countingSortByDigit(v, (int)place);
    }
}

int main() {
    vector<int> nums{170, 45, 75, 90, 802, 24, 2, 66};
    radixSort(nums);
    cout << "radix sorted: ";
    for (int x : nums) cout << x << " ";
    cout << "\n";

    // WHY EACH PASS MUST BE STABLE: consider 45 and 75 -- both have ones digit 5. After
    // the ones-digit pass, a stable sort keeps them ordered relative to each other based
    // on whichever appeared first in the original array. The NEXT pass (tens digit) sorts
    // by a DIFFERENT digit, but for elements that end up with the SAME tens digit, it must
    // PRESERVE whatever relative order the previous pass established -- otherwise the
    // correct ones-digit ordering achieved earlier gets silently destroyed. An unstable
    // digit-sort would produce an incorrectly ordered final result.

    // COMPLEXITY: O(d * (n + k)) where d is the number of digits and k is the base (10
    // here) -- for fixed-width integers, d is a constant, so this is effectively O(n),
    // beating comparison sorts' O(n log n) floor, the same way counting sort does, just
    // applied digit-by-digit to handle a value range too large for counting sort directly

    return 0;
}


In [ ]:
!g++ -std=c++17 -O2 -Wall -o temp temp.cpp && ./temp

## 7. Sorting Stability — What It Means and When It Matters

### The Why
Stability sounds like a minor implementation detail until you're sorting by one key while relying on a DIFFERENT, previously-established order to survive for elements that tie on the new key — at which point it's the difference between a correct and an incorrect result, as radix sort just demonstrated concretely.

### Core Mechanism
- **Definition:** a sort is stable if elements that compare EQUAL keep their original relative order after sorting.
- **Where this matters practically:** sorting a list of objects by one field (say, grade) while wanting objects with the SAME grade to remain in whatever order they had before (say, arrival order) — this is a multi-key sort achieved through a single stable pass, rather than needing an explicit tiebreaker comparator.
- **Which algorithms are stable by construction:** merge sort (Section 2) — the `<=` comparison in its merge step always prefers the left half on ties. Counting sort and radix sort (Sections 5–6), when implemented as shown, are also stable.
- **Which are NOT stable in typical implementations:** quicksort and heap sort — their in-place swapping can reorder equal elements arbitrarily, with no guarantee preserved. `std::sort` (the STL general-purpose sort) is explicitly NOT guaranteed stable, since its introsort implementation (Section 8) prioritizes speed over this guarantee — use `std::stable_sort` (`03_STL_Deep_Dive`, Section 11) whenever tie-breaking order matters.

### Common Pitfalls
- Relying on `std::sort`'s apparent behavior looking stable on a particular compiler or input size, then having that assumption silently break on a different compiler, standard library version, or input — `std::sort` makes NO stability guarantee, and "it happened to work" is not the same as "it's guaranteed to work." Use `std::stable_sort` explicitly whenever the property is actually required.


In [ ]:
%%writefile temp.cpp
#include <iostream>
#include <vector>
#include <algorithm>
using namespace std;

// STABILITY: a sort is STABLE if elements that compare EQUAL keep their original
// relative order after sorting. This matters whenever you sort by one key but the
// original order encoded some OTHER meaningful information you don't want scrambled.

struct Student {
    string name;
    int grade;
};

int main() {
    // sorted by insertion order (arrival), NOT yet by grade
    vector<Student> students{
        {"Alice", 90}, {"Bob", 85}, {"Carol", 90}, {"Dave", 85}, {"Eve", 95}
    };

    // stable_sort by grade: students with the SAME grade keep their original relative
    // order -- Bob still comes before Dave, Alice still comes before Carol, because
    // that's how they appeared in the original array
    vector<Student> stableSorted = students;
    stable_sort(stableSorted.begin(), stableSorted.end(),
                [](const Student& a, const Student& b) { return a.grade < b.grade; });

    cout << "stable_sort result (by grade):\n";
    for (auto& s : stableSorted) cout << "  " << s.name << ": " << s.grade << "\n";
    // Bob(85), Dave(85), Alice(90), Carol(90), Eve(95) -- Bob before Dave, Alice before Carol

    // regular sort makes NO such guarantee -- same-grade students COULD end up reordered
    // (implementation-dependent; may or may not visibly differ on a given run/compiler,
    // which is exactly what makes relying on plain sort() for this purpose dangerous --
    // it might happen to look stable today and break tomorrow with a different compiler
    // or a different-sized input)

    // WHICH BUILT-IN SORTS ARE STABLE: merge sort (this notebook's Section 3) is stable
    // BY CONSTRUCTION, since the merge step's "<=" comparison always prefers the left
    // half on ties. std::sort is explicitly NOT guaranteed stable (its introsort
    // implementation prioritizes speed over this guarantee) -- use std::stable_sort
    // whenever tie-breaking order matters. Quicksort and heap sort are NOT stable in
    // their typical in-place implementations, because their swapping can reorder equal
    // elements arbitrarily.

    return 0;
}


In [ ]:
!g++ -std=c++17 -O2 -Wall -o temp temp.cpp && ./temp

## 8. Choosing the Right Sort

### The Why
This closes the loop on everything above: given a NEW situation, which of these six algorithms (plus `std::sort`/`std::stable_sort` from the STL) actually fits? "Just use `std::sort`" is the right default in most everyday code — but knowing when it isn't is what this whole notebook has been building toward.

### Core Mechanism
- **Default choice for general-purpose code:** `std::sort` — it's introsort: quicksort as the primary strategy for speed, falling back to heap sort if recursion depth suggests a bad quicksort case (bounding the worst case at O(n log n) rather than risking true O(n²)), and switching to insertion sort for small sub-arrays (exploiting Section 1's near-sorted-data advantage). This is a genuinely well-engineered hybrid, not an arbitrary pick — it's worth knowing you're getting three algorithms' strengths combined, not just "the standard library's sort."
- **When you need a stability guarantee:** `std::stable_sort` — merge-sort-based, guaranteed O(n log n), guaranteed stable, at the cost of O(n) extra space versus introsort's typically smaller footprint.
- **When the value range is small and known:** counting sort (Section 5) — O(n+k), beats any comparison sort when k is genuinely small relative to n.
- **When values have bounded digit-width but too large a range for direct counting sort:** radix sort (Section 6).
- **When memory is severely constrained and a strict O(n log n) worst-case guarantee matters more than typical-case speed:** heap sort (Section 4) — genuinely the right choice in memory-constrained embedded contexts, even though it's rarely fastest in general-purpose code.
- **When the data is small (roughly under a few dozen elements) or nearly sorted already:** insertion sort (Section 1) directly — this is exactly why `std::sort` already does this for you internally, but it's worth knowing explicitly for cases outside a generic library call.

### Common Pitfalls
- Hand-implementing quicksort or merge sort in production code "for practice" when `std::sort`/`std::stable_sort` already provide a more thoroughly tested, better-engineered version — implement these by hand for LEARNING (as this notebook does), but reach for the STL in real code, same guidance as `03_STL_Deep_Dive`'s heap section.


## 9. Phase Checkpoint, Cheat Sheet, and Answer Key

### Sorting Algorithm Cheat Sheet

| Algorithm | Best | Average | Worst | Space | Stable? | Notes |
|---|---|---|---|---|---|---|
| Bubble Sort | O(n) | O(n²) | O(n²) | O(1) | Yes | Early-exit on already-sorted input |
| Selection Sort | O(n²) | O(n²) | O(n²) | O(1) | No (typical impl.) | No early exit possible; minimal swaps |
| Insertion Sort | O(n) | O(n²) | O(n²) | O(1) | Yes | Fast on nearly-sorted data; real-world small-array choice |
| Merge Sort | O(n log n) | O(n log n) | O(n log n) | O(n) | Yes | Guaranteed, but not in-place |
| Quick Sort | O(n log n) | O(n log n) | O(n²) | O(log n) avg (stack) | No (typical impl.) | Fast in practice; needs good pivot strategy |
| Heap Sort | O(n log n) | O(n log n) | O(n log n) | O(1) | No | Guaranteed + in-place, but poor cache locality |
| Counting Sort | O(n+k) | O(n+k) | O(n+k) | O(n+k) | Yes | Only for small integer ranges |
| Radix Sort | O(d(n+k)) | O(d(n+k)) | O(d(n+k)) | O(n+k) | Yes (requires stable per-digit sort) | For bounded-width integers with too-large a range for direct counting sort |

### Checkpoint Questions
1. Why does insertion sort's best case matter in practice, given it's still O(n²) worst case?
2. State merge sort's recurrence relation and explain, in one or two sentences, why it resolves to O(n log n).
3. What specific input makes the Lomuto quicksort partition (pivot = last element) hit its O(n²) worst case, and what fixes it?
4. Heap sort matches merge sort's worst-case guarantee AND matches quicksort's space efficiency — why isn't it the default choice for general-purpose sorting anyway?
5. What structural assumption about the data does counting sort require, and why does violating it (large value range) make the algorithm impractical even though its formula still "works"?
6. Why must radix sort's per-digit sorting passes be stable, specifically — what breaks if they aren't?

### Answer Key
1. Real-world data is often already sorted or nearly sorted (appending to a mostly-sorted log, re-sorting after a small change). Insertion sort's near-sorted performance is close to O(n), dramatically better than its worst-case O(n²) — this is why it's the practical choice for small or mostly-ordered data, and why general-purpose sorts like introsort switch to it for small sub-arrays.
2. `T(n) = 2·T(n/2) + O(n)`. Two half-size recursive calls (the divide) plus O(n) work to merge the results back together. There are `log n` levels of recursion (halving repeatedly until reaching size 1), and the total merge work at any single level is O(n) — combining `log n` levels × O(n) work per level gives O(n log n).
3. An already-sorted (or reverse-sorted) input, when the pivot is always chosen as the last element — every partition produces a maximally unbalanced split (0 elements on one side, n-1 on the other). The fix is choosing the pivot randomly (or via median-of-three), which makes this specific adversarial structure no longer special to the algorithm.
4. Despite matching merge sort's time guarantee and quicksort's space efficiency on paper, heap sort has poor cache locality — its heap-index-based access pattern jumps around memory rather than accessing it sequentially, unlike merge sort's linear merges or quicksort's typically-local partitioning. This usually makes it measurably slower in practice, which is why real implementations (like introsort) use it only as a worst-case safety net, not the primary strategy.
5. Counting sort requires values to be integers (or map to small integer keys) within a KNOWN, REASONABLY SMALL range. Violating this (e.g. sorting values up to 10⁹) makes the count array allocation itself impractical — the O(n+k) formula is technically accurate but `k` dominates and makes the actual resource cost enormous, even though the algorithm is still "correct."
6. If a per-digit pass isn't stable, elements that tie on the CURRENT digit can get reordered arbitrarily relative to each other — destroying the correct relative ordering established by PREVIOUS (less significant digit) passes. This isn't a minor imperfection; it produces a genuinely incorrectly-sorted final result, since later passes depend on earlier passes' orderings surviving intact for tied elements.

### Mini Project
Benchmark all three O(n log n) algorithms from this notebook (merge sort, quicksort, heap sort) against `std::sort` on the same randomly generated data, using `<chrono>` from `01_Complexity_Analysis`.
1. Generate a few different input types: random data, already-sorted data, and reverse-sorted data, at a consistent size (e.g. n=100,000).
2. Time each algorithm on each input type, being careful to use the `volatile`-result technique from `01_Complexity_Analysis` if needed to prevent the compiler from optimizing work away.
3. Confirm empirically: quicksort should be fast on random data but noticeably slower (or you may need a larger n to see it clearly) on sorted/reverse-sorted input with the fixed last-element pivot from Section 3 — compare this against `std::sort`'s introsort, which should stay fast across all three input types due to its fallback mechanism.

This exercises: the `<chrono>` benchmarking skill from `01_Complexity_Analysis`, and turns this notebook's theoretical claims about quicksort's worst case and introsort's hybrid design into something empirically measured rather than taken on faith.


## 10. LeetCode Practice Problems

Sorting rarely appears as "implement a sort" directly on LeetCode — it's almost always a TOOL inside a larger problem. These are grouped by how sorting gets used, which is the more realistic skill to practice.

### Direct Sorting Implementation

| # | Problem | Difficulty | Hint |
|---|---|---|---|
| 912 | Sort an Array | Medium | Implement any O(n log n) sort from this notebook directly — this problem exists specifically to make you write one from scratch |
| 148 | Sort List | Medium | Merge sort's divide-and-conquer structure applied to a linked list instead of an array — you'll revisit this in `07_Linked_Lists` |

### Sort-Then-Two-Pointer (Combining With Phase 02's Earlier Notebooks)

| # | Problem | Difficulty | Hint |
|---|---|---|---|
| 15 | 3Sum | Medium | Already in `01_Arrays_and_Two_Pointers`' mini project — sort first, THEN two pointers; listed here as the canonical "sort enables a different pattern" example |
| 56 | Merge Intervals | Medium | Sort intervals by start time first — once sorted, overlapping intervals become adjacent and easy to merge in one pass |
| 435 | Non-overlapping Intervals | Medium | Sort by END time (not start) — a greedy interval-scheduling problem that depends entirely on choosing the right sort key |

### Counting Sort / Bucket-Style Thinking

| # | Problem | Difficulty | Hint |
|---|---|---|---|
| 75 | Sort Colors | Medium | Already covered via Dutch National Flag in `01_Arrays_and_Two_Pointers` — also directly solvable with a two-pass counting sort (count 0s/1s/2s, then overwrite); compare both approaches |
| 347 | Top K Frequent Elements | Medium | Bucket sort by frequency count (bucket index = frequency, since frequency is bounded by array length) can beat a heap-based O(n log k) approach with O(n) instead |

### Custom Comparators

| # | Problem | Difficulty | Hint |
|---|---|---|---|
| 179 | Largest Number | Medium | Sort strings with a custom comparator: `a+b > b+a` (string concatenation comparison) determines the ordering — not a numeric comparison at all |
| 973 | K Closest Points to Origin | Medium | Sort by squared distance (avoid an unnecessary sqrt), or use a heap for a better-than-full-sort complexity when k is much smaller than n |

### Self-Check Before Moving to Phase 03
If you can look at a new problem and recognize "this needs sorting FIRST, then a different technique from Phase 02" as a two-step plan — rather than reaching for sorting as the whole solution — you've internalized sorting's real role in this course: not usually the final answer, but the setup step that makes an earlier pattern (two pointers, binary search) applicable. Phase 03 moves into hierarchical structures (trees), where you'll see recursion and traversal ideas from Phase 01 combine with everything built here.
